[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/multimodal_online_evaluation/multimodal_online_evaluation.ipynb)

# Multimodal online evaluation

This guide shows how to run an **online LLM-as-judge evaluation over multimodal
(text + image) traces** in Opik. You will:

1. Log traces that carry an image plus text, structured so an online rule can map them.
2. Create the online-evaluation rule two ways — in the Opik UI, and with the SDK.
3. Run the rule and review the feedback scores.

Along the way it highlights the **best practices for defining image variables** in
an online-evaluation rule: add them with the **"Images +"** button (rather than
typing `{{image_output}}` into the prompt), use a **vision-capable model**, and keep
score definitions independent of variable mapping.

In [ ]:
import os

import opik

OPIK_API_KEY = os.environ.get("OPIK_API_KEY")
OPIK_WORKSPACE = os.environ.get("OPIK_WORKSPACE")
OPIK_URL = os.environ.get("OPIK_URL_OVERRIDE", "https://www.comet.com/opik/api")

DRY_RUN = not (OPIK_API_KEY and OPIK_WORKSPACE)

PROJECT_NAME = os.environ.get("OPIK_PROJECT_NAME", "multimodal-online-evaluation")

PUBLIC_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
# WHY: a 1x1 transparent PNG keeps the notebook small while still exercising the base64 path.
BASE64_IMAGE_URL = (
    "data:image/png;base64,"
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=="
)

client = None if DRY_RUN else opik.Opik(project_name=PROJECT_NAME)

if DRY_RUN:
    print("[DRY RUN] OPIK_API_KEY / OPIK_WORKSPACE not set — will print payloads instead of sending.")
else:
    print(f"Connected to Opik at {OPIK_URL}; logging to project '{PROJECT_NAME}'.")

## How the traces are structured

Each trace keeps the image URL in the **input**, alongside the question, and the
text answer plus expected answer in the **output**:

- `input.question` — the request
- `input.image_url` — the image (public URL or base64 data URI)
- `output.answer` — the model's answer
- `output.expected_answer` — the reference answer

The online rule maps its variables to these paths. Keeping the image in
`input.image_url` means one mapping works for every trace.

In [ ]:
def build_trace_payload(image_url: str) -> dict:
    return {
        "name": "describe-image",
        "input": {
            "question": "Describe the animal in the attached image.",
            "image_url": image_url,
        },
        "output": {
            "answer": "The image shows a domestic cat sitting upright.",
            "expected_answer": "A cat.",
        },
        "tags": ["multimodal-online-evaluation"],
    }


payloads = [build_trace_payload(PUBLIC_IMAGE_URL), build_trace_payload(BASE64_IMAGE_URL)]

if DRY_RUN:
    for p in payloads:
        shown = {**p, "input": {**p["input"], "image_url": p["input"]["image_url"][:60] + "..."}}
        print("[DRY RUN] would log trace:", shown)
else:
    for p in payloads:
        client.trace(**p)
    client.flush()
    print(f"Logged {len(payloads)} traces to project '{PROJECT_NAME}'.")

## Creating the online-evaluation rule

There are two ways to create the rule that scores these traces. They are
different paths with different benefits — pick the one that fits; both produce
the same rule.

- **In the Opik UI** — the fastest way to try it and see it visually. Best when
  you are exploring or setting a rule up once.
- **With the SDK** — reproducible and version-controllable. Best when the rule
  should live in code or run in CI.

### Create it in the UI

1. Open the `multimodal-online-evaluation` project → **Online evaluation** tab → **Create new rule**.
2. Choose **LLM-as-judge** and select a **vision-capable model** (e.g. `gpt-4o` or `claude-3-5-sonnet`). Without a vision model the image is dropped to a text placeholder.
3. Paste the judge prompt below. The text variables use `{{ }}`; the image is added separately in the next step.

    ```text
    You are an impartial evaluator scoring how accurately a model described an image.

    You are given the user's request, the model's text answer, and an expected
    reference answer. An image is also attached; it is the image the model was asked
    to describe.

    Request:
    {{input}}

    Model answer:
    {{output}}

    Expected answer:
    {{expected_output}}

    Judge whether the model's answer accurately and completely describes the
    attached image, and whether it is consistent with the expected answer.

    Score the following:
    - accuracy_score: how accurate the model's answer is with respect to the
      attached image and the expected answer, as an integer 1 (completely
      inaccurate) to 5 (fully accurate).
    - describes_image_correctly: true if the model's answer correctly identifies
      the main subject of the attached image, false otherwise.

    Return only the structured result.
    ```

4. Add the image variable with the **"Images +"** button. **Do not** type `{{image_output}}` into the prompt text.

    > **Note:** Referencing the same image variable via the button *and* in the prompt text registers it twice, which makes the variable mapping and score definitions look duplicated. Use the button only.

5. Map the variables:

    | Variable | Maps to |
    |---|---|
    | `input` | `input.question` |
    | `output` | `output.answer` |
    | `expected_output` | `output.expected_answer` |
    | `image_output` | `input.image_url` |

6. Add the score definitions:

    | Name | Type | Description |
    |---|---|---|
    | `accuracy_score` | Integer | Accuracy of the answer vs. the attached image and the expected answer, 1-5. |
    | `describes_image_correctly` | Boolean | Whether the answer correctly identifies the main subject of the attached image. |

    > **Note:** Score definitions are independent of variable mapping. A score with no mapped variable still returns a judge-inferred value, so map the image via "Images +" — don't rely on the score definition alone.

7. Save. Private/attachment image URLs must be reachable by the model provider; the public URL and base64 data URI in this guide both work.

### Create the same rule with the SDK

The cell below builds the identical rule programmatically. Note the image is an
`image_url` **content part** in `content_array` — the SDK equivalent of the
"Images +" button — not a `{{ }}` text variable.

In [ ]:
from opik.rest_api import (
    AutomationRuleEvaluatorWrite_LlmAsJudge,
    ImageUrlWrite,
    LlmAsJudgeCodeWrite,
    LlmAsJudgeMessageContentWrite,
    LlmAsJudgeMessageWrite,
    LlmAsJudgeModelParametersWrite,
    LlmAsJudgeOutputSchemaWrite,
)

JUDGE_PROMPT = """You are an impartial evaluator scoring how accurately a model described an image.

You are given the user's request, the model's text answer, and an expected
reference answer. An image is also attached; it is the image the model was asked
to describe.

Request:
{{input}}

Model answer:
{{output}}

Expected answer:
{{expected_output}}

Judge whether the model's answer accurately and completely describes the
attached image, and whether it is consistent with the expected answer.

Score the following:
- accuracy_score: how accurate the model's answer is with respect to the
  attached image and the expected answer, as an integer 1 (completely
  inaccurate) to 5 (fully accurate).
- describes_image_correctly: true if the model's answer correctly identifies
  the main subject of the attached image, false otherwise.

Return only the structured result."""

VARIABLES = {
    "input": "input.question",
    "output": "output.answer",
    "expected_output": "output.expected_answer",
    "image_output": "input.image_url",
}

judge_message = LlmAsJudgeMessageWrite(
    role="USER",
    # WHY: the image is an image_url content part (the SDK equivalent of the UI
    # "Images +" button), NOT a {{ }} text variable.
    content_array=[
        LlmAsJudgeMessageContentWrite(type="text", text=JUDGE_PROMPT),
        LlmAsJudgeMessageContentWrite(
            type="image_url", image_url=ImageUrlWrite(url="{{image_output}}")
        ),
    ],
    structured_content=True,
)

code = LlmAsJudgeCodeWrite(
    model=LlmAsJudgeModelParametersWrite(name="gpt-4o", temperature=0.0),
    messages=[judge_message],
    variables=VARIABLES,
    schema_=[
        LlmAsJudgeOutputSchemaWrite(
            name="accuracy_score",
            type="INTEGER",
            description="Accuracy vs. the attached image and expected answer, 1-5.",
        ),
        LlmAsJudgeOutputSchemaWrite(
            name="describes_image_correctly",
            type="BOOLEAN",
            description="Whether the answer identifies the main subject of the image.",
        ),
    ],
)

if DRY_RUN:
    print("[DRY RUN] would create rule 'multimodal-accuracy' with code:", code)
else:
    # WHY: get_project() looks up by project id; retrieve_project(name=...) resolves
    # WHY: the just-created project by its name to get the id for project_ids.
    project = client.rest_client.projects.retrieve_project(name=PROJECT_NAME)
    request = AutomationRuleEvaluatorWrite_LlmAsJudge(
        name="multimodal-accuracy",
        project_ids=[project.id],
        sampling_rate=1.0,
        enabled=True,
        action="evaluator",
        code=code,
    )
    client.rest_client.automation_rule_evaluators.create_automation_rule_evaluator(request=request)
    print("Created online-evaluation rule 'multimodal-accuracy'.")

## Run the rule and review scores

An online-evaluation rule scores traces two ways:

- **Automatically, on new traces** — once the rule is enabled it evaluates traces as they are logged, sampled at the rule's sampling rate (`1.0` = every trace). Log a new trace to the project and its scores appear shortly after.
- **Manually, on existing traces** — the two traces above were logged *before* the rule existed, so score them on demand from the **Traces** tab: select the traces and trigger the rule from the trace toolbar.

Either way, results land as **feedback scores** (`accuracy_score`, `describes_image_correctly`) on each trace and roll up in the project dashboard.
